# Inno4Vac example workflows

Consolidated from the standalone notebooks in this folder for the Inno4Vac annual meeting:
- **Part 1 — USD centrifugation workflow** (from `usd_example.ipynb`, which already includes everything `usd_config.ipynb` set up, plus a worked mock trial and comparison plots).
- **Part 2 — USP bioreactor CFD compartmentalisation** (from `USP.ipynb`).

Both are upstream/harvest-process analysis workflows (ultra scale-down centrifugation and bioreactor mixing) and don't configure a chromatography model, so there is no natural `cadetgui` widget to embed here. For a `cadetgui`-widget-based (downstream, chromatography) example, see `examples/configuration_and_solution.ipynb` or `examples/workbench.ipynb` instead.

# Part 1: USD centrifugation workflow

In [1]:
import pandas as pd
import numpy as np

# This DataFrame will hold all runs (example data and your own entries)
runs = pd.DataFrame(
    columns=[
        "run_id",
        "shear_level",
        "tube_type",
        "pilot_condition",
        "volume_mL",
        "rpm",
        "spin_time_min",
        "flow_Lph",
        "yield_percent",
        "clarity_NTU",
        "pellet_volume_mL",
        "supernatant_OD600",
        "endotoxin_UA_per_mL",
        "solids_remaining_frac",
        "sigma_lab_m2",
        "flow_m3_per_s",
        "Q_over_sigma_m_per_s",
        "delta_OD600_vs_pilot",
        "endotoxin_reduction_vs_feed",
        "delta_solids_vs_pilot"
    ]
)

runs


,run_id,shear_level,tube_type,pilot_condition,volume_mL,rpm,spin_time_min,flow_Lph,yield_percent,clarity_NTU,pellet_volume_mL,supernatant_OD600,endotoxin_UA_per_mL,solids_remaining_frac,sigma_lab_m2,flow_m3_per_s,Q_over_sigma_m_per_s,delta_OD600_vs_pilot,endotoxin_reduction_vs_feed,delta_solids_vs_pilot


## Configuration for shear levels and tube types

You can edit the tables in this cell to match your experiment.


In [2]:
# Shear configuration
SHEAR_CONFIG = pd.DataFrame([
    {
        "shear_level": "Well Spun",
        "kompas_rpm": None,
        "default_flow_Lph": 100.0,
        "description": "No additional shear (reference condition)"
    },
    {
        "shear_level": "Low",
        "kompas_rpm": 6000,
        "default_flow_Lph": 100.0,
        "description": "Low shear, equivalent to hydrohermetic centrifuge"
    },
    {
        "shear_level": "Medium",
        "kompas_rpm": 7500,
        "default_flow_Lph": 100.0,
        "description": "Medium shear condition"
    },
    {
        "shear_level": "High",
        "kompas_rpm": 9000,
        "default_flow_Lph": 100.0,
        "description": "High shear, non hydrohermetic equivalent"
    }
])

# Tube configuration
TUBE_CONFIG = pd.DataFrame([
    {
        "tube_type": "2 mL Eppendorf",
        "max_volume_mL": 2.0,
        "min_volume_mL": 1.0,
        "description": "Standard 2 mL microcentrifuge tube"
    },
    {
        "tube_type": "1.5 mL Eppendorf",
        "max_volume_mL": 1.5,
        "min_volume_mL": 0.5,
        "description": "Standard 1.5 mL microcentrifuge tube"
    }
])

print("Shear levels:")
display(SHEAR_CONFIG)

print("Tube types:")
display(TUBE_CONFIG)


Shear levels:


,shear_level,kompas_rpm,default_flow_Lph,description
0,Well Spun,NaN,100.0,No additional shear (reference condition)
1,Low,6000.0,100.0,"Low shear, equivalent to hydrohermetic centrifuge"
2,Medium,7500.0,100.0,Medium shear condition
3,High,9000.0,100.0,"High shear, non hydrohermetic equivalent"


Tube types:


,tube_type,max_volume_mL,min_volume_mL,description
0,2 mL Eppendorf,2.0,1.0,Standard 2 mL microcentrifuge tube
1,1.5 mL Eppendorf,1.5,0.5,Standard 1.5 mL microcentrifuge tube


## Pilot reference values

Enter pilot scale reference values here.

Defaults below are set to a mock example similar to your Sanofi trial:
- Feed OD600 = 8.1
- Pilot clarification fraction = 0.878 (about 88 percent)
- Feed endotoxin = 3 260 000 UA/mL

Below that you can define pilot condition specific data such as solids remaining or pilot supernatant OD600 per condition.


In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from cadetgui.widgets.elements import FloatField, ChoiceField, TextField

# Global pilot inputs (edit as needed)
pilot_feed_OD600_widget = FloatField(label="Feed OD600", value=8.1, min=0.0)

pilot_percent_clar_widget = FloatField(
    label="Pilot clarification fraction", value=0.878, min=0.0, max=1.0
)

pilot_feed_endotoxin_widget = FloatField(
    label="Feed endotoxin", value=3260000.0, units="UA/mL", min=0.0
)

pilot_box = widgets.VBox([
    pilot_feed_OD600_widget,
    pilot_percent_clar_widget,
    pilot_feed_endotoxin_widget
])

display(pilot_box)

# Pilot condition specific data (edit this table to match your pilot runs)
# Example mock values:
# Q1: slightly worse than global pilot
# Q2: similar
# Q3: slightly better clarification

PILOT_CONDITIONS = pd.DataFrame([
    {
        "pilot_condition": "Q1",
        "solids_remaining_frac": 0.16,
        "pilot_supernatant_OD600": 1.3,
        "pilot_pellet_wet_mass_g": 0.35
    },
    {
        "pilot_condition": "Q2",
        "solids_remaining_frac": 0.14,
        "pilot_supernatant_OD600": 1.1,
        "pilot_pellet_wet_mass_g": 0.37
    },
    {
        "pilot_condition": "Q3",
        "solids_remaining_frac": 0.12,
        "pilot_supernatant_OD600": 0.9,
        "pilot_pellet_wet_mass_g": 0.40
    }
])

print("Pilot condition table (edit values in this cell if needed):")
PILOT_CONDITIONS


Pilot condition table (edit values in this cell if needed):


,pilot_condition,solids_remaining_frac,pilot_supernatant_OD600,pilot_pellet_wet_mass_g
0,Q1,0.16,1.3,0.35
1,Q2,0.14,1.1,0.37
2,Q3,0.12,0.9,0.40


## Centrifuge parameters and Sigma helper

You can adjust these values to match your bench top centrifuge and rotor.


In [4]:
# Physical constant
g = 9.81  # m s^-2

# Bench top centrifuge configuration (adjust as needed)
CENTRIFUGE_CONFIG = {
    "name": "Eppendorf 5424R",
    "rotor": "F45-24-11",
    "r0": 0.040,
    "L_tube": 0.040,
    "theta_deg": 45.0,
    "h_gel": 0.011,
    "t_acc": 15.0,
    "t_dec": 16.0
}

def fractional_time(total_time_s, acc_s, dec_s):
    f_acc = acc_s / total_time_s
    f_dec = dec_s / total_time_s
    return f_acc, f_dec

def r_top_and_bottom(r0, L_tube, h_gel, theta_deg):
    """Simplified geometry based on tube length and agar height.

    You can replace this with expressions from your exact Eq. 4 and 5.
    """
    theta = np.deg2rad(theta_deg)
    r_top = r0 + (L_tube - h_gel) * np.sin(theta)
    r_bottom = r0 + L_tube * np.sin(theta)
    return r_top, r_bottom

def sigma_lab(V_m3, rpm, spin_time_s, config=CENTRIFUGE_CONFIG, correction_factor=1.0):
    """Compute an approximate equivalent settling area Sigma of the lab centrifuge.

    You can refine this function once you decide on the exact Eq. 3 you want to use.
    """
    omega = 2 * np.pi * rpm / 60.0
    f_acc, f_dec = fractional_time(spin_time_s, config["t_acc"], config["t_dec"])
    r_top, r_bottom = r_top_and_bottom(config["r0"], config["L_tube"], config["h_gel"], config["theta_deg"])

    sigma = (
        omega**2
        * V_m3
        * (1.0 - f_acc - f_dec)
        * (r_bottom**2 - r_top**2)
        / (2.0 * g * np.log(r_bottom / r_top))
    )

    return correction_factor * sigma


## Mock USD example trial

This cell creates a small mock USD data set with a few runs at different shear levels and Q values, linked to pilot conditions Q1 to Q3.

Run it once to populate `runs` with example data. You can still add more runs via the widgets afterward.


In [5]:
def create_mock_example_runs():
    """Create a small mock USD trial and compute all derived quantities."""
    example_rows = []

    # Read pilot global values
    feed_OD600 = pilot_feed_OD600_widget.value
    pilot_clar = pilot_percent_clar_widget.value
    feed_endotoxin = pilot_feed_endotoxin_widget.value

    # Helper to compute derived metrics for one run
    def one_run(run_id, shear_level, tube_type, pilot_condition,
                volume_mL, rpm, spin_time_min, flow_Lph,
                yield_percent, clarity_NTU, pellet_volume_mL,
                supernatant_OD600, endotoxin_UA_per_mL,
                solids_remaining_frac):

        V_m3 = volume_mL * 1e-6
        t_s = spin_time_min * 60.0
        sigma_val = sigma_lab(V_m3, rpm, t_s)

        if flow_Lph is not None:
            flow_m3_per_s = flow_Lph / 1000.0 / 3600.0
            Q_over_sigma = flow_m3_per_s / sigma_val if sigma_val not in [0, np.nan] else np.nan
        else:
            flow_m3_per_s = np.nan
            Q_over_sigma = np.nan

        # Global pilot OD600 comparison
        if (
            supernatant_OD600 is not None
            and pd.notna(feed_OD600)
            and pd.notna(pilot_clar)
        ):
            pilot_supernatant_OD600_global = feed_OD600 * (1.0 - pilot_clar)
            delta_OD600_vs_pilot = supernatant_OD600 - pilot_supernatant_OD600_global
        else:
            delta_OD600_vs_pilot = np.nan

        # Endotoxin reduction vs feed
        if (
            endotoxin_UA_per_mL is not None
            and pd.notna(feed_endotoxin)
            and feed_endotoxin != 0
        ):
            endotoxin_reduction_vs_feed = 1.0 - endotoxin_UA_per_mL / feed_endotoxin
        else:
            endotoxin_reduction_vs_feed = np.nan

        # Pilot condition specific solids comparison
        if pilot_condition is not None and pilot_condition != "":
            pilot_row = PILOT_CONDITIONS.set_index("pilot_condition").loc[pilot_condition]
            pilot_solids_remaining_frac = pilot_row["solids_remaining_frac"]
        else:
            pilot_solids_remaining_frac = np.nan

        if solids_remaining_frac is not None and pd.notna(pilot_solids_remaining_frac):
            delta_solids_vs_pilot = solids_remaining_frac - pilot_solids_remaining_frac
        else:
            delta_solids_vs_pilot = np.nan

        return {
            "run_id": run_id,
            "shear_level": shear_level,
            "tube_type": tube_type,
            "pilot_condition": pilot_condition,
            "volume_mL": volume_mL,
            "rpm": rpm,
            "spin_time_min": spin_time_min,
            "flow_Lph": flow_Lph,
            "yield_percent": yield_percent,
            "clarity_NTU": clarity_NTU,
            "pellet_volume_mL": pellet_volume_mL,
            "supernatant_OD600": supernatant_OD600,
            "endotoxin_UA_per_mL": endotoxin_UA_per_mL,
            "solids_remaining_frac": solids_remaining_frac,
            "sigma_lab_m2": sigma_val,
            "flow_m3_per_s": flow_m3_per_s,
            "Q_over_sigma_m_per_s": Q_over_sigma,
            "delta_OD600_vs_pilot": delta_OD600_vs_pilot,
            "endotoxin_reduction_vs_feed": endotoxin_reduction_vs_feed,
            "delta_solids_vs_pilot": delta_solids_vs_pilot
        }

    # Example runs (2 mL, 5 min, 100 L/h), three shear levels and three pilot conditions
    example_rows.append(one_run(
        run_id="mock1",
        shear_level="Well Spun",
        tube_type="2 mL Eppendorf",
        pilot_condition="Q1",
        volume_mL=2.0,
        rpm=6000,
        spin_time_min=5.0,
        flow_Lph=100.0,
        yield_percent=96.0,
        clarity_NTU=20.0,
        pellet_volume_mL=0.32,
        supernatant_OD600=1.4,
        endotoxin_UA_per_mL=900000.0,
        solids_remaining_frac=0.18
    ))

    example_rows.append(one_run(
        run_id="mock2",
        shear_level="Low",
        tube_type="2 mL Eppendorf",
        pilot_condition="Q2",
        volume_mL=2.0,
        rpm=6000,
        spin_time_min=5.0,
        flow_Lph=100.0,
        yield_percent=94.0,
        clarity_NTU=18.0,
        pellet_volume_mL=0.34,
        supernatant_OD600=1.1,
        endotoxin_UA_per_mL=750000.0,
        solids_remaining_frac=0.15
    ))

    example_rows.append(one_run(
        run_id="mock3",
        shear_level="High",
        tube_type="2 mL Eppendorf",
        pilot_condition="Q3",
        volume_mL=2.0,
        rpm=9000,
        spin_time_min=5.0,
        flow_Lph=100.0,
        yield_percent=90.0,
        clarity_NTU=16.0,
        pellet_volume_mL=0.36,
        supernatant_OD600=0.85,
        endotoxin_UA_per_mL=650000.0,
        solids_remaining_frac=0.13
    ))

    # Second replicate set to have some variation
    example_rows.append(one_run(
        run_id="mock1_rep",
        shear_level="Well Spun",
        tube_type="2 mL Eppendorf",
        pilot_condition="Q1",
        volume_mL=2.0,
        rpm=6000,
        spin_time_min=5.0,
        flow_Lph=100.0,
        yield_percent=95.0,
        clarity_NTU=21.0,
        pellet_volume_mL=0.31,
        supernatant_OD600=1.5,
        endotoxin_UA_per_mL=920000.0,
        solids_remaining_frac=0.19
    ))

    example_rows.append(one_run(
        run_id="mock2_rep",
        shear_level="Low",
        tube_type="2 mL Eppendorf",
        pilot_condition="Q2",
        volume_mL=2.0,
        rpm=6000,
        spin_time_min=5.0,
        flow_Lph=100.0,
        yield_percent=93.0,
        clarity_NTU=19.0,
        pellet_volume_mL=0.35,
        supernatant_OD600=1.2,
        endotoxin_UA_per_mL=780000.0,
        solids_remaining_frac=0.16
    ))

    example_rows.append(one_run(
        run_id="mock3_rep",
        shear_level="High",
        tube_type="2 mL Eppendorf",
        pilot_condition="Q3",
        volume_mL=2.0,
        rpm=9000,
        spin_time_min=5.0,
        flow_Lph=100.0,
        yield_percent=89.0,
        clarity_NTU=15.0,
        pellet_volume_mL=0.37,
        supernatant_OD600=0.80,
        endotoxin_UA_per_mL=640000.0,
        solids_remaining_frac=0.12
    ))

    df = pd.DataFrame(example_rows)
    return df

# Create example data and store it in `runs`
runs = create_mock_example_runs()
runs


,run_id,shear_level,tube_type,pilot_condition,volume_mL,rpm,spin_time_min,flow_Lph,yield_percent,clarity_NTU,pellet_volume_mL,supernatant_OD600,endotoxin_UA_per_mL,solids_remaining_frac,sigma_lab_m2,flow_m3_per_s,Q_over_sigma_m_per_s,delta_OD600_vs_pilot,endotoxin_reduction_vs_feed,delta_solids_vs_pilot
0,mock1,Well Spun,2 mL Eppendorf,Q1,2.0,6000,5.0,100.0,96.0,20.0,0.32,1.40,900000.0,0.18,0.000299,0.000028,0.092933,0.4118,0.723926,0.02
1,mock2,Low,2 mL Eppendorf,Q2,2.0,6000,5.0,100.0,94.0,18.0,0.34,1.10,750000.0,0.15,0.000299,0.000028,0.092933,0.1118,0.769939,0.01
2,mock3,High,2 mL Eppendorf,Q3,2.0,9000,5.0,100.0,90.0,16.0,0.36,0.85,650000.0,0.13,0.000673,0.000028,0.041303,-0.1382,0.800613,0.01
3,mock1_rep,Well Spun,2 mL Eppendorf,Q1,2.0,6000,5.0,100.0,95.0,21.0,0.31,1.50,920000.0,0.19,0.000299,0.000028,0.092933,0.5118,0.717791,0.03
4,mock2_rep,Low,2 mL Eppendorf,Q2,2.0,6000,5.0,100.0,93.0,19.0,0.35,1.20,780000.0,0.16,0.000299,0.000028,0.092933,0.2118,0.760736,0.02
5,mock3_rep,High,2 mL Eppendorf,Q3,2.0,9000,5.0,100.0,89.0,15.0,0.37,0.80,640000.0,0.12,0.000673,0.000028,0.041303,-0.1882,0.803681,0.00


## Widget based input for additional runs

Use the widgets to enter further USD runs and click "Add run".

These runs will be appended to the existing example data in `runs`.


In [6]:
# Widgets for USD runs (cadetgui elements: FloatField/ChoiceField/TextField)

# 1. Run information and settings
run_id_text = TextField(label="Run ID", value="user1")

shear_dropdown = ChoiceField(
    label="Shear",
    options=[(v, v) for v in SHEAR_CONFIG["shear_level"].tolist()],
)

tube_dropdown = ChoiceField(
    label="Tube",
    options=[(v, v) for v in TUBE_CONFIG["tube_type"].tolist()],
)

pilot_condition_dropdown = ChoiceField(
    label="Pilot condition",
    options=[("(none)", "")] + [(c, c) for c in PILOT_CONDITIONS["pilot_condition"].tolist()],
)

volume_float = FloatField(label="Volume", value=2.0, units="mL", min=0.0)

rpm_float = FloatField(label="Speed", value=6000.0, units="rpm", min=0.0)

spin_time_float = FloatField(label="Spin time", value=5.0, units="min", min=0.0)

flow_float = FloatField(label="Flow", value=100.0, units="L/h", min=0.0)

# 2. Measured results -- optional, 0 means "not measured". FloatField's value
# trait has no NaN-safe JSON serializer (unlike ipywidgets.FloatText), so a
# real sentinel (0) is used instead of NaN to keep these visually consistent
# with the rest of the form.
supernatant_OD600_float = FloatField(label="Supernatant OD600 (0 = n/a)", value=0.0, min=0.0)

solids_remaining_float = FloatField(
    label="Solids remaining frac (0 = n/a)", value=0.0, units="frac", min=0.0, max=1.0
)

endotoxin_run_float = FloatField(label="Endotoxin (0 = n/a)", value=0.0, units="UA/mL", min=0.0)

pellet_float = FloatField(label="Pellet volume (0 = n/a)", value=0.0, units="mL", min=0.0)

yield_float = FloatField(label="Yield (0 = n/a)", value=0.0, units="%", min=0.0, max=100.0)

clarity_float = FloatField(label="Clarity (0 = n/a)", value=0.0, units="NTU", min=0.0)

add_button = widgets.Button(
    description="Add run",
    button_style="success",
    layout=widgets.Layout(width="150px")
)

output_last_run = widgets.Output()

def update_volume_bounds(*args):
    tube = tube_dropdown.value
    row = TUBE_CONFIG.set_index("tube_type").loc[tube]
    max_v = row["max_volume_mL"]
    min_v = row["min_volume_mL"]
    if volume_float.value > max_v:
        volume_float.value = max_v
    if volume_float.value < min_v:
        volume_float.value = min_v

# ChoiceField's real synced trait is selected_index, not value -- see
# ai-docs/ARCHITECTURE.md's note on this exact gotcha.
tube_dropdown.observe(update_volume_bounds, names=tube_dropdown._value_trait_name)

def on_add_button_clicked(b):
    global runs
    with output_last_run:
        clear_output()

        run_id = run_id_text.value.strip() or "user_run"
        shear_level = shear_dropdown.value
        tube_type = tube_dropdown.value
        pilot_condition = pilot_condition_dropdown.value or None

        volume_mL = float(volume_float.value)
        rpm = float(rpm_float.value)
        spin_time_min = float(spin_time_float.value)
        flow_Lph = float(flow_float.value) if not np.isnan(flow_float.value) else None

        # Optional measured outputs -- 0 means "not measured" (see field setup above)
        yield_percent = yield_float.value or None
        clarity_NTU = clarity_float.value or None
        pellet_volume_mL = pellet_float.value or None
        supernatant_OD600 = supernatant_OD600_float.value or None
        endotoxin_run = endotoxin_run_float.value or None
        solids_remaining = solids_remaining_float.value or None

        # Derived quantities
        V_m3 = volume_mL * 1e-6
        t_s = spin_time_min * 60.0
        sigma_val = sigma_lab(V_m3, rpm, t_s)

        if flow_Lph is not None:
            flow_m3_per_s = flow_Lph / 1000.0 / 3600.0
            Q_over_sigma = flow_m3_per_s / sigma_val if sigma_val not in [0, np.nan] else np.nan
        else:
            flow_m3_per_s = np.nan
            Q_over_sigma = np.nan

        feed_OD600 = pilot_feed_OD600_widget.value
        pilot_clar = pilot_percent_clar_widget.value
        feed_endotoxin = pilot_feed_endotoxin_widget.value

        if (
            supernatant_OD600 is not None
            and pd.notna(feed_OD600)
            and pd.notna(pilot_clar)
        ):
            pilot_supernatant_OD600_global = feed_OD600 * (1.0 - pilot_clar)
            delta_OD600_vs_pilot = supernatant_OD600 - pilot_supernatant_OD600_global
        else:
            delta_OD600_vs_pilot = np.nan

        if (
            endotoxin_run is not None
            and pd.notna(feed_endotoxin)
            and feed_endotoxin != 0
        ):
            endotoxin_reduction_vs_feed = 1.0 - endotoxin_run / feed_endotoxin
        else:
            endotoxin_reduction_vs_feed = np.nan

        if pilot_condition is not None and pilot_condition != "":
            pilot_row = PILOT_CONDITIONS.set_index("pilot_condition").loc[pilot_condition]
            pilot_solids_remaining_frac = pilot_row["solids_remaining_frac"]
        else:
            pilot_solids_remaining_frac = np.nan

        if solids_remaining is not None and pd.notna(pilot_solids_remaining_frac):
            delta_solids_vs_pilot = solids_remaining - pilot_solids_remaining_frac
        else:
            delta_solids_vs_pilot = np.nan

        row = {
            "run_id": run_id,
            "shear_level": shear_level,
            "tube_type": tube_type,
            "pilot_condition": pilot_condition,
            "volume_mL": volume_mL,
            "rpm": rpm,
            "spin_time_min": spin_time_min,
            "flow_Lph": flow_Lph,
            "yield_percent": yield_percent,
            "clarity_NTU": clarity_NTU,
            "pellet_volume_mL": pellet_volume_mL,
            "supernatant_OD600": supernatant_OD600,
            "endotoxin_UA_per_mL": endotoxin_run,
            "solids_remaining_frac": solids_remaining,
            "sigma_lab_m2": sigma_val,
            "flow_m3_per_s": flow_m3_per_s,
            "Q_over_sigma_m_per_s": Q_over_sigma,
            "delta_OD600_vs_pilot": delta_OD600_vs_pilot,
            "endotoxin_reduction_vs_feed": endotoxin_reduction_vs_feed,
            "delta_solids_vs_pilot": delta_solids_vs_pilot
        }

        runs = pd.concat([runs, pd.DataFrame([row])], ignore_index=True)

        print("Run added:")
        display(pd.DataFrame([row]))

add_button.on_click(on_add_button_clicked)

# Layout with clear separation of inputs and outputs
form_left = widgets.VBox([
    widgets.HTML("<b>1. Run information</b>"),
    run_id_text,
    shear_dropdown,
    tube_dropdown,
    pilot_condition_dropdown,
    widgets.HTML("<b>2. Centrifugation settings</b>"),
    volume_float,
    rpm_float,
    spin_time_float,
    flow_float
])

form_right = widgets.VBox([
    widgets.HTML("<b>3. Measured results (optional)</b>"),
    supernatant_OD600_float,
    solids_remaining_float,
    endotoxin_run_float,
    pellet_float,
    yield_float,
    clarity_float,
    add_button
])

form = widgets.HBox([form_left, form_right])

display(form)
display(output_last_run)


Output()

## View all runs (example plus any additional user runs)


In [7]:
runs


,run_id,shear_level,tube_type,pilot_condition,volume_mL,rpm,spin_time_min,flow_Lph,yield_percent,clarity_NTU,pellet_volume_mL,supernatant_OD600,endotoxin_UA_per_mL,solids_remaining_frac,sigma_lab_m2,flow_m3_per_s,Q_over_sigma_m_per_s,delta_OD600_vs_pilot,endotoxin_reduction_vs_feed,delta_solids_vs_pilot
0,mock1,Well Spun,2 mL Eppendorf,Q1,2.0,6000,5.0,100.0,96.0,20.0,0.32,1.40,900000.0,0.18,0.000299,0.000028,0.092933,0.4118,0.723926,0.02
1,mock2,Low,2 mL Eppendorf,Q2,2.0,6000,5.0,100.0,94.0,18.0,0.34,1.10,750000.0,0.15,0.000299,0.000028,0.092933,0.1118,0.769939,0.01
2,mock3,High,2 mL Eppendorf,Q3,2.0,9000,5.0,100.0,90.0,16.0,0.36,0.85,650000.0,0.13,0.000673,0.000028,0.041303,-0.1382,0.800613,0.01
3,mock1_rep,Well Spun,2 mL Eppendorf,Q1,2.0,6000,5.0,100.0,95.0,21.0,0.31,1.50,920000.0,0.19,0.000299,0.000028,0.092933,0.5118,0.717791,0.03
4,mock2_rep,Low,2 mL Eppendorf,Q2,2.0,6000,5.0,100.0,93.0,19.0,0.35,1.20,780000.0,0.16,0.000299,0.000028,0.092933,0.2118,0.760736,0.02
5,mock3_rep,High,2 mL Eppendorf,Q3,2.0,9000,5.0,100.0,89.0,15.0,0.37,0.80,640000.0,0.12,0.000673,0.000028,0.041303,-0.1882,0.803681,0.00


## Summary tables and averages

This cell computes averages per shear level and tube type.


In [8]:
metrics = [
    "yield_percent",
    "clarity_NTU",
    "pellet_volume_mL",
    "Q_over_sigma_m_per_s",
    "supernatant_OD600",
    "delta_OD600_vs_pilot",
    "endotoxin_UA_per_mL",
    "endotoxin_reduction_vs_feed",
    "solids_remaining_frac",
    "delta_solids_vs_pilot"
]

group_cols = ["shear_level", "tube_type"]

if not runs.empty:
    summary = (
        runs
        .groupby(group_cols)[metrics]
        .agg(["mean", "std", "count"])
        .sort_index()
    )
    summary
else:
    print("No runs recorded yet.")


## Summary per shear level only


In [9]:
metrics = [
    "yield_percent",
    "clarity_NTU",
    "pellet_volume_mL",
    "Q_over_sigma_m_per_s",
    "supernatant_OD600",
    "delta_OD600_vs_pilot",
    "endotoxin_UA_per_mL",
    "endotoxin_reduction_vs_feed",
    "solids_remaining_frac",
    "delta_solids_vs_pilot"
]

if not runs.empty:
    summary_shear = (
        runs
        .groupby("shear_level")[metrics]
        .agg(["mean", "std", "count"])
        .sort_index()
    )
    summary_shear
else:
    print("No runs recorded yet.")


## Plots: USD vs pilot

The following cells create a few meaningful plots for this workflow, as interactive Plotly figures (hover over a point/bar to inspect its exact value).

In [10]:
import plotly.express as px

# Q/Sigma vs solids remaining, no pilot overlay -- see the next cells for pilot comparisons
if runs.empty:
    print("No data in 'runs' to plot.")
else:
    df_plot = runs[["run_id", "Q_over_sigma_m_per_s", "solids_remaining_frac", "shear_level"]].dropna(
        subset=["Q_over_sigma_m_per_s", "solids_remaining_frac"]
    )
    if df_plot.empty:
        print("No complete Q/Sigma and solids data to plot.")
    else:
        fig = px.scatter(
            df_plot,
            x="Q_over_sigma_m_per_s",
            y="solids_remaining_frac",
            color="shear_level",
            hover_data=["run_id"],
            labels={
                "Q_over_sigma_m_per_s": "Q/Sigma (m/s)",
                "solids_remaining_frac": "Solids remaining fraction",
                "shear_level": "Shear level",
            },
            title="Q/Sigma vs solids remaining",
        )
        fig.show()


In [11]:
import plotly.graph_objects as go

if runs.empty:
    print("No data in 'runs' to plot. Run the mock example cell or add runs via the widgets first.")
else:
    feed_OD600 = pilot_feed_OD600_widget.value
    pilot_clar = pilot_percent_clar_widget.value
    pilot_sup_OD = (
        feed_OD600 * (1.0 - pilot_clar) if pd.notna(feed_OD600) and pd.notna(pilot_clar) else np.nan
    )

    od_summary = runs.groupby("shear_level")["supernatant_OD600"].agg(["mean", "std"]).dropna(subset=["mean"])

    if od_summary.empty:
        print("No supernatant OD600 values available for plotting.")
    else:
        fig = go.Figure(go.Bar(
            x=od_summary.index,
            y=od_summary["mean"],
            error_y=dict(type="data", array=od_summary["std"].fillna(0)),
        ))
        if not np.isnan(pilot_sup_OD):
            fig.add_hline(y=pilot_sup_OD, line_dash="dash", annotation_text="pilot")
        fig.update_layout(
            title="USD supernatant OD600 by shear level",
            xaxis_title="Shear level",
            yaxis_title="Supernatant OD600",
        )
        fig.show()


In [12]:
import plotly.graph_objects as go

if runs.empty:
    print("No data in 'runs' to plot.")
else:
    et_summary = runs.groupby("shear_level")["endotoxin_reduction_vs_feed"].agg(["mean", "std"]).dropna(subset=["mean"])

    if et_summary.empty:
        print("No endotoxin reduction values available for plotting.")
    else:
        fig = go.Figure(go.Bar(
            x=et_summary.index,
            y=et_summary["mean"],
            error_y=dict(type="data", array=et_summary["std"].fillna(0)),
        ))
        fig.update_layout(
            title="Endotoxin reduction by shear level",
            xaxis_title="Shear level",
            yaxis_title="Mean endotoxin reduction vs feed",
            yaxis_range=[0, 1],
        )
        fig.show()


In [13]:
import plotly.graph_objects as go

if runs.empty:
    print("No data in 'runs' to plot.")
else:
    pilot_solids = PILOT_CONDITIONS[["pilot_condition", "solids_remaining_frac"]].rename(
        columns={"solids_remaining_frac": "pilot_solids_frac"}
    )
    df_merge = runs.merge(pilot_solids, on="pilot_condition", how="left")

    lab_means = df_merge.groupby("pilot_condition")["solids_remaining_frac"].mean().dropna()
    pilot_vals = df_merge.groupby("pilot_condition")["pilot_solids_frac"].first().dropna()

    common_conds = sorted(set(lab_means.index) & set(pilot_vals.index))
    if not common_conds:
        print("No overlapping pilot_condition entries with both pilot and lab solids.")
    else:
        lab_means = lab_means.loc[common_conds]
        pilot_vals = pilot_vals.loc[common_conds]

        fig = go.Figure()
        fig.add_bar(x=common_conds, y=pilot_vals.values, name="pilot")
        fig.add_bar(x=common_conds, y=lab_means.values, name="USD (lab)")
        fig.update_layout(
            barmode="group",
            title="Pilot vs USD solids remaining per pilot condition",
            xaxis_title="Pilot condition",
            yaxis_title="Solids remaining fraction",
        )
        fig.show()


In [14]:
import plotly.express as px
import plotly.graph_objects as go

if runs.empty:
    print("No data in 'runs' to plot.")
else:
    pilot_solids = PILOT_CONDITIONS[["pilot_condition", "solids_remaining_frac"]].rename(
        columns={"solids_remaining_frac": "pilot_solids_frac"}
    )
    df_merge = runs.merge(pilot_solids, on="pilot_condition", how="left")

    df_plot = df_merge[
        ["run_id", "Q_over_sigma_m_per_s", "solids_remaining_frac", "shear_level", "pilot_condition", "pilot_solids_frac"]
    ].dropna(subset=["Q_over_sigma_m_per_s", "solids_remaining_frac"])

    if df_plot.empty:
        print("No complete Q/Sigma and solids data to plot.")
    else:
        fig = px.scatter(
            df_plot,
            x="Q_over_sigma_m_per_s",
            y="solids_remaining_frac",
            color="shear_level",
            hover_data=["run_id"],
            labels={
                "Q_over_sigma_m_per_s": "Q/Sigma (m/s)",
                "solids_remaining_frac": "Solids remaining fraction",
                "shear_level": "Shear level",
            },
        )
        # Pilot solids as its own hoverable horizontal-line trace per pilot condition
        for cond, sub in df_plot.groupby("pilot_condition"):
            if pd.isna(cond) or cond == "":
                continue
            pilot_val = sub["pilot_solids_frac"].iloc[0]
            q_min = sub["Q_over_sigma_m_per_s"].min()
            q_max = sub["Q_over_sigma_m_per_s"].max()
            fig.add_trace(go.Scatter(
                x=[q_min, q_max],
                y=[pilot_val, pilot_val],
                mode="lines",
                name=f"{cond} pilot",
                line=dict(dash="dash"),
            ))

        fig.update_layout(title="Q/Sigma vs solids remaining\nUSD predictions and pilot levels")
        fig.show()


# Part 2: USP bioreactor CFD compartmentalisation

In [15]:
%%capture
import sys
!{sys.executable} -m pip install cbsm
!{sys.executable} -m pip install pandas
!{sys.executable} -m pip install jupyterlab-spreadsheet-editor

In [16]:
import os
from cbsm.compartmentalisation import CM
import pandas as pd
import os
print(os.getcwd())

/home/IBT/lanzrath/CADET/CADET-GUI/examples/inno4vac


In [17]:
df = pd.read_csv(os.getcwd()+"/"+"/2_A310_60_BM_160rpm_FlowStats_003.csv")
df.to_excel(os.getcwd()+"/"+"/2_A310_60_BM_160rpm_FlowStats_003.xlsx", index=False)

In [18]:
df

,X [ m ],Y [ m ],Z [ m ],Area [ m^2 ],Gas.Volume Fraction.Trnavg,kla [ s^-1 ],Liquid.Velocity.Trnavg Axial [ m s^-1 ],Liquid.Velocity.Trnavg Radial [ m s^-1 ],Ostar [ kg m^-3 ],Radius [ m ]
0,0.008728,0.000366,-0.008728,0.002751,0.016278,0.007243,0.002616,0.005892,0.020083,0.015292
1,0.004302,0.000392,-0.004302,0.003260,0.016267,0.007241,0.002619,0.008496,0.020085,0.017024
2,0.080751,0.003482,-0.080751,0.004477,0.016231,0.007817,0.000231,0.049452,0.020095,0.114443
3,-0.081709,0.003634,0.081709,0.006354,0.015869,0.007640,-0.004847,0.125770,0.020127,0.144917
4,0.128352,0.005612,-0.128352,0.001925,0.016190,0.008139,-0.001209,0.081542,0.020103,0.183175
...,...,...,...,...,...,...,...,...,...,...
20553,1.069738,5.205605,-1.069738,0.001263,0.165631,0.078018,0.142636,0.059292,0.018081,1.512878
20554,-1.152214,5.205691,1.152214,0.000276,0.148606,0.068185,0.019984,0.008445,0.018105,1.629534
20555,-0.982000,5.206203,0.982000,0.001181,0.188362,0.092127,0.261627,0.238571,0.018101,1.388794
20556,-0.137613,5.206447,0.137613,0.000569,0.181688,0.094480,0.121910,0.003036,0.018025,0.195211


In [19]:
import plotly.express as px

# X/Y and per-point flow speed from the first two + two velocity-component columns
x = df.iloc[:, 0]
y = df.iloc[:, 1]
speed = np.hypot(df.iloc[:, 6], df.iloc[:, 7])

# A per-point hoverable scatter trades the continuous gouraud-shaded mesh for
# real point-level inspection (exact speed under the cursor).
fig = px.scatter(
    x=x, y=y, color=speed,
    color_continuous_scale="Viridis",
    labels={"x": "X [m]", "y": "Y [m]", "color": "Speed [m/s]"},
    title="CFD flow speed",
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()


In [ ]:
filename = "/2_A310_60_BM_160rpm_FlowStats_003.xlsx"
# Read CFD data
data = pd.read_excel(os.getcwd()+"/"+filename)
CFD_data = data.values
# Extract data
# start at 5 to avoid the header
X = CFD_data[1:, 9].tolist()
Y = CFD_data[1:, 1].tolist()
Vx = CFD_data[1:, 7].tolist()
Vy = CFD_data[1:, 6].tolist()

(grid_info, volume, img) = CM(X, Y, Vx, Vy)


In [ ]:
from cbsm import splitting as splitting
import numpy as np


center_info = np.array([[1.0, 1.0, 1.0], [1.5, 1.5, 1.0], [3.0, 3.0, 1.0]])
delta_x = 1.0
delta_y = 1.0

split_result = splitting.split(center_info, delta_x, delta_y)


In [ ]:
split_result